# Earthquake Damage Classification in Nepal Using Machine Learning

**Module:** CSC61304 – Machine Learning and Intelligent Systems  
**Project task:** Predict whether a building is likely to experience severe earthquake damage using information recorded before the earthquake.

## 1. Problem definition and objectives

Earthquakes can cause extensive damage to buildings and disrupt communities. A machine-learning model may help disaster-management teams identify building characteristics associated with severe damage and support prioritisation for inspection and preparedness. It must be treated as a decision-support tool, not as a replacement for structural engineering assessment.

This notebook has five objectives:

1. Load and describe the Nepal earthquake building-damage dataset.
2. Explore damage patterns using clear, labelled visualisations.
3. Clean the data and prevent target leakage by excluding post-earthquake information.
4. Build and compare two models: Logistic Regression and Decision Tree.
5. Tune each model, evaluate them on an untouched test set, and interpret the results.

**Prediction target:** `severe_damage = 1` for damage grades 4–5, and `0` for grades 1–3. The binary target is used to make the classification task easier to interpret for a potential screening system.

## 2. Dataset source and workflow

The dataset is the Nepal earthquake building-damage dataset distributed through the Kaggle project *Earthquake Damage in Nepal*. In Kaggle, add the dataset to the notebook before running the code. The expected file name is `NepalEarhquakeDamage2015.csv` (note the spelling in the source file name).

The main workflow is:

**Dataset → data inspection → remove leakage/identifier columns → create target → EDA → stratified train/validation/test split → preprocessing → baseline models → validation-based tuning → final test evaluation → interpretation.**

The code is intentionally kept close to a beginner-friendly notebook style. It uses standard pandas, seaborn, matplotlib, and scikit-learn tools rather than extra encoding packages.

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 3. Load the data

The file search below supports the usual Kaggle input folder and a local copy in the working directory. If the file is not found, check that the Kaggle dataset has been added and that its filename matches the expected name.

In [ ]:
file_name = "NepalEarhquakeDamage2015.csv"
data_path = None

for folder in ["/kaggle/input", ".", ".."]:
    if os.path.exists(folder):
        for root, _, files in os.walk(folder):
            if file_name in files:
                data_path = os.path.join(root, file_name)
                break
    if data_path:
        break

if data_path is None:
    raise FileNotFoundError(
        f"Could not find {file_name}. Add the dataset to Kaggle or update data_path."
    )

df = pd.read_csv(data_path)
print("Dataset shape:", df.shape)
df.head()

### 3.1 Understand the dataset

Inspect the column names, data types, missing values, and summary statistics before making changes. This helps document the dataset and identify data-quality issues.

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(15))
df.describe(include="all").T.head(20)

## 4. Data cleaning and target creation

`damage_grade` is the outcome to be predicted. Grades 4 and 5 are grouped as severe damage (1), while grades 1, 2, and 3 are grouped as not severe (0). The original `damage_grade` column is then removed so the answer is not included among the predictors.

Columns containing `post_eq` describe conditions after the earthquake. They are removed because the intended system should make a prediction using pre-earthquake or otherwise available information. `building_id` is an identifier rather than a building characteristic, so it is also removed.

The district identifier is retained as a categorical predictor, allowing the model to learn regional differences. These choices should be documented and justified in the report.

In [ ]:
# Keep a record of the feature-selection decisions.
# These rules use domain knowledge and prevent obvious leakage.
identifier_cols = [col for col in ["building_id"] if col in df.columns]
post_eq_cols = [col for col in df.columns if "post_eq" in col.lower()]

# Convert damage grade to a numeric value and remove rows without a valid target.
df["damage_grade"] = pd.to_numeric(
    df["damage_grade"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)
rows_before = len(df)
df = df.dropna(subset=["damage_grade"]).copy()
df = df[df["damage_grade"].isin([1, 2, 3, 4, 5])].copy()
rows_removed_invalid_target = rows_before - len(df)

# Remove identifiers and post-earthquake fields before creating predictors.
# Post-earthquake fields are excluded because they would not be available
# when making a pre-event or early screening prediction.
df = df.drop(columns=identifier_cols + post_eq_cols, errors="ignore")

# Binary target: grades 4–5 = severe (1); grades 1–3 = not severe (0).
df["severe_damage"] = (df["damage_grade"] >= 4).astype(int)
df = df.drop(columns=["damage_grade"])

print("Identifier columns removed:", identifier_cols)
print("Post-earthquake columns removed:", post_eq_cols)
print("Rows removed because target grade was missing/invalid:", rows_removed_invalid_target)
print("Cleaned dataset shape:", df.shape)
print("Remaining missing values (top 10):")
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing_count"))
df.head()


## 5. Exploratory Data Analysis (EDA)

EDA is used to understand the target distribution and explore possible relationships between building characteristics and severe damage. The plots below should be described in the report using the actual patterns visible after running the notebook. Avoid claiming that a feature causes damage based only on these plots.

### 5.1 Class balance

The class-balance plot shows how many records belong to each target class. This matters because accuracy can be misleading when one class is much more common than the other.

In [ ]:
print(df["severe_damage"].value_counts())
print(df["severe_damage"].value_counts(normalize=True).round(3))

sns.countplot(data=df, x="severe_damage")
plt.title("Figure 1. Distribution of severe and non-severe damage")
plt.xlabel("Severe damage (0 = no, 1 = yes)")
plt.ylabel("Number of buildings")
plt.show()

### 5.2 Numeric feature distributions

Histograms provide a quick view of the distribution of numeric building characteristics and may reveal skewness or unusual values.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop("severe_damage", errors="ignore")
df[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.suptitle("Figure 2. Distributions of numeric building features")
plt.tight_layout()
plt.show()

### 5.3 Compare selected building characteristics by damage class

The boxplot compares building height across the two target classes. Use it to discuss differences in distributions, overlap, and possible outliers.

In [ ]:
if "height_ft_pre_eq" in df.columns:
    sns.boxplot(data=df, x="severe_damage", y="height_ft_pre_eq")
    plt.title("Figure 3. Building height by severe-damage class")
    plt.xlabel("Severe damage (0 = no, 1 = yes)")
    plt.ylabel("Building height before earthquake (feet)")
    plt.show()
else:
    print("height_ft_pre_eq is not available in this dataset version.")

### 5.4 Severe-damage rate by foundation type

This plot compares the proportion of severe-damage cases across foundation categories. Categories with few records should be interpreted cautiously.

In [ ]:
if "foundation_type" in df.columns:
    foundation_rate = df.groupby("foundation_type")["severe_damage"].mean().sort_values()
    foundation_rate.plot(kind="barh", figsize=(8, 5))
    plt.title("Figure 4. Proportion of severe damage by foundation type")
    plt.xlabel("Proportion with severe damage")
    plt.ylabel("Foundation type")
    plt.show()
    display(foundation_rate.rename("severe_damage_rate").to_frame())
else:
    print("foundation_type is not available in this dataset version.")

## 6. Prepare features and split the data

The target is separated from the input features. A stratified split keeps approximately the same class proportions in each subset:

- **70% training set:** fits the model.
- **15% validation set:** compares hyperparameter choices and selects the model.
- **15% test set:** used once for the final, unbiased comparison.

The test set must not be used to choose model settings. This helps prevent overly optimistic performance estimates.

In [ ]:
X = df.drop(columns="severe_damage")
y = df["severe_damage"]

# First reserve 15% for final testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)

# Split the remaining 85% so validation is 15% of the full dataset
validation_size = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=validation_size,
    random_state=RANDOM_STATE, stratify=y_train_val
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)
print("Training target proportions:")
print(y_train.value_counts(normalize=True).sort_index().round(3))

### 6.1 Preprocessing and feature selection

**Feature selection (rule-based):** `building_id` is removed because it identifies a record rather than describing a building. Columns containing `post_eq` and the original `damage_grade` are removed to prevent target leakage. The remaining pre-earthquake building characteristics and available location fields are retained. This is a transparent, domain-based selection method; it is not a claim that every retained feature is equally useful. Feature importance is examined later as a diagnostic.

**Missing values:** numeric columns use median imputation; categorical columns use the most frequent category. **Encoding:** categorical variables are one-hot encoded so their labels are not treated as ordered numbers. **Scaling:** numeric values are standardised for Logistic Regression. Decision Trees do not require scaling, but the same preprocessing design is used for a fair, consistent workflow.

The train/validation/test split is made before fitting preprocessing. Each model pipeline learns imputation values, scaling parameters, and category encodings from its training data only. This avoids information leakage from validation or test data.


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

def make_preprocessor():
    """Create a fresh preprocessing pipeline for each model."""
    numeric_preprocessing = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_preprocessing = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("numeric", numeric_preprocessing, numeric_features),
        ("categorical", categorical_preprocessing, categorical_features)
    ])

print("Selected numeric features:", len(numeric_features))
print("Selected categorical features:", len(categorical_features))
print("Total selected predictor columns:", len(numeric_features) + len(categorical_features))
print("Predictor columns:")
print(X_train.columns.tolist())


## 7. Baseline

A majority-class baseline always predicts the most common class in the training set. Its accuracy provides a simple reference point. A useful model should be assessed with more than accuracy, because failing to identify severe damage may have important consequences.

In [ ]:
baseline_accuracy = y_train.value_counts(normalize=True).max()
print("Majority-class baseline accuracy:", round(baseline_accuracy, 3))

## 8. Model 1 — Logistic Regression

Logistic Regression is a linear classification model. It provides a relatively simple benchmark and can estimate the probability of severe damage. The initial model uses a regularisation strength of `C=1.0`; this value will be tuned using the validation set.

In [ ]:
log_model = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced",
                                      random_state=RANDOM_STATE))
])

log_model.fit(X_train, y_train)
log_val_pred = log_model.predict(X_val)

print("Validation accuracy:", round(accuracy_score(y_val, log_val_pred), 3))
print("Validation macro F1:", round(f1_score(y_val, log_val_pred, average="macro"), 3))

### 8.1 Tune Logistic Regression

The regularisation parameter `C` controls the strength of regularisation. We test a small, clearly documented set of values and select the value with the highest validation macro F1-score. Macro F1 gives equal weight to each class.

In [ ]:
log_c_values = [0.01, 0.1, 1, 10]
log_tuning_results = []

for c in log_c_values:
    candidate = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("classifier", LogisticRegression(
            C=c, max_iter=1000, class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    score = f1_score(y_val, val_pred, average="macro")
    log_tuning_results.append({"C": c, "validation_macro_f1": score})

log_tuning_results = pd.DataFrame(log_tuning_results)
display(log_tuning_results)

best_log_c = log_tuning_results.loc[
    log_tuning_results["validation_macro_f1"].idxmax(), "C"
]
print("Best Logistic Regression C:", best_log_c)

## 9. Model 2 — Decision Tree

A Decision Tree learns a sequence of feature-based rules and can model non-linear relationships. Trees can overfit if they grow too deep, so `max_depth` is tuned using validation macro F1 rather than selecting a depth from training accuracy alone.

In [ ]:
tree_model = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", DecisionTreeClassifier(
        random_state=RANDOM_STATE, class_weight="balanced"
    ))
])

tree_model.fit(X_train, y_train)
tree_val_pred = tree_model.predict(X_val)

print("Validation accuracy:", round(accuracy_score(y_val, tree_val_pred), 3))
print("Validation macro F1:", round(f1_score(y_val, tree_val_pred, average="macro"), 3))

### 9.1 Tune Decision Tree

We compare a small range of maximum tree depths. A shallow tree is easier to interpret, while a very deep tree may memorise the training data. The best depth is chosen using validation macro F1, and the test set remains untouched.

In [ ]:
tree_depths = [3, 5, 8, 12, 16, None]
tree_tuning_results = []

for depth in tree_depths:
    candidate = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("classifier", DecisionTreeClassifier(
            max_depth=depth, random_state=RANDOM_STATE,
            class_weight="balanced"
        ))
    ])
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    score = f1_score(y_val, val_pred, average="macro")
    tree_tuning_results.append({
        "max_depth": str(depth),
        "validation_macro_f1": score
    })

tree_tuning_results = pd.DataFrame(tree_tuning_results)
display(tree_tuning_results)

best_depth_text = tree_tuning_results.loc[
    tree_tuning_results["validation_macro_f1"].idxmax(), "max_depth"
]
best_tree_depth = None if best_depth_text == "None" else int(best_depth_text)
print("Best Decision Tree max_depth:", best_tree_depth)

tree_tuning_results.plot(
    x="max_depth", y="validation_macro_f1", marker="o", legend=False
)
plt.title("Figure 5. Decision Tree depth tuning")
plt.xlabel("Maximum depth")
plt.ylabel("Validation macro F1-score")
plt.show()

### 9.2 Select the model using validation data only

Before looking at test-set results, compare the best Logistic Regression and Decision Tree settings using their **validation macro F1-scores**. The model with the higher validation macro F1 is the model selected for the proposed system. The test set is not used to choose the model or its hyperparameters.

Macro F1 is the primary selection metric because it gives equal weight to the severe and not-severe classes. Also inspect severe-class recall: a model that misses many severely damaged buildings may be unsuitable for a screening-support system, even if its overall accuracy looks good.


In [ ]:
# Get the best validation score from each tuning experiment.
best_log_validation_f1 = float(
    log_tuning_results["validation_macro_f1"].max()
)
best_tree_validation_f1 = float(
    tree_tuning_results["validation_macro_f1"].max()
)

validation_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Best validation macro F1": [
        best_log_validation_f1, best_tree_validation_f1
    ]
}).set_index("Model")

display(validation_comparison.round(3))
validation_comparison.plot(kind="bar", legend=False, ylim=(0, 1), figsize=(7, 4))
plt.title("Figure 6. Model selection using validation macro F1")
plt.ylabel("Validation macro F1-score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

if best_log_validation_f1 >= best_tree_validation_f1:
    selected_model_name = "Logistic Regression"
else:
    selected_model_name = "Decision Tree"

print("Model selected using validation data:", selected_model_name)


## 10. Final test-set evaluation

After model settings and the preferred model have been selected using validation data, refit each tuned model on the combined training and validation sets. Evaluate both models on the held-out test set for a transparent comparison. **Do not change the selected model or tune settings based on test-set results.**

Report accuracy, macro precision, macro recall, macro F1, and ROC-AUC. Also inspect severe-class precision and recall, confusion matrices, and the precision-recall curve. Macro F1 is the primary metric used during model selection; severe-class recall is an important safety-related diagnostic because missed severe cases could be consequential. These predictions are decision support, not a substitute for structural inspection by qualified professionals.


In [ ]:
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

final_log_model = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", LogisticRegression(
        C=float(best_log_c), max_iter=1000, class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

final_tree_model = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", DecisionTreeClassifier(
        max_depth=best_tree_depth, random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

final_log_model.fit(X_train_final, y_train_final)
final_tree_model.fit(X_train_final, y_train_final)

models = {
    "Logistic Regression": final_log_model,
    "Decision Tree": final_tree_model
}

results = []
for name, model in models.items():
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test, pred, average="macro", zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })

results = pd.DataFrame(results).set_index("Model")
display(results.round(3))

### 10.1 Confusion matrices and classification reports

A confusion matrix shows correct and incorrect predictions for each class. The classification report provides precision, recall, and F1-score for each class. Compare false negatives (severe buildings predicted as non-severe) carefully, as they are particularly relevant to a disaster-screening use case.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=["Not severe", "Severe"],
        cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(f"Figure 6. {name} confusion matrix")
plt.tight_layout()
plt.show()

for name, model in models.items():
    print("\n", name)
    print(classification_report(
        y_test, model.predict(X_test),
        target_names=["Not severe", "Severe"], zero_division=0
    ))

### 10.2 ROC and precision-recall curves

These curves compare model discrimination across classification thresholds. The precision-recall curve is particularly informative when classes are imbalanced. The final interpretation should refer to the curves and the numerical metrics actually produced by the notebook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=axes[1], name=name)

axes[0].set_title("Figure 7. ROC curves on the test set")
axes[1].set_title("Figure 8. Precision-recall curves on the test set")
plt.tight_layout()
plt.show()

### 10.3 Compare test metrics and interpret the selected model

The chart compares the test metrics for both models. Use the validation result above to identify the model selected for the proposed system; test results describe final generalisation performance and must not be used to re-select the model. In the report, discuss overall performance, severe-class recall, false negatives, and the trade-off between performance and interpretability.


In [ ]:
# The selected model was chosen using validation macro F1, not test scores.
print("Model selected before test evaluation:", selected_model_name)
print("\\nTest-set metrics for both models (reporting only):")
display(results.round(3))

selected_test_prediction = models[selected_model_name].predict(X_test)
print("\\nClassification report for the validation-selected model:")
print(classification_report(
    y_test,
    selected_test_prediction,
    target_names=["Not severe", "Severe"],
    zero_division=0
))


## 11. Model interpretation

Feature importance and coefficients can help explain what the models learned, but they do not prove that a feature causes damage. One-hot encoding means a single original categorical variable may become several model features. Interpret these results alongside domain knowledge and the dataset's limitations.

### 11.1 Decision Tree feature importance

In [ ]:
tree_preprocessor = final_tree_model.named_steps["preprocessor"]
tree_classifier = final_tree_model.named_steps["classifier"]
tree_feature_names = tree_preprocessor.get_feature_names_out()

tree_importance = pd.Series(
    tree_classifier.feature_importances_, index=tree_feature_names
).sort_values(ascending=False)

display(tree_importance.head(15).to_frame("importance"))
tree_importance.head(15).sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Figure 10. Top Decision Tree feature importances")
plt.xlabel("Gini importance")
plt.ylabel("Encoded feature")
plt.show()

### 11.2 Logistic Regression coefficients

In [ ]:
log_preprocessor = final_log_model.named_steps["preprocessor"]
log_classifier = final_log_model.named_steps["classifier"]
log_feature_names = log_preprocessor.get_feature_names_out()

log_coefficients = pd.Series(
    log_classifier.coef_[0], index=log_feature_names
).sort_values()

print("Features associated with lower predicted odds of severe damage:")
display(log_coefficients.head(10).to_frame("coefficient"))

print("Features associated with higher predicted odds of severe damage:")
display(log_coefficients.tail(10).sort_values(ascending=False).to_frame("coefficient"))

## 12. Proposed intelligent-system architecture

The diagram below presents the five main elements of the proposed intelligent decision-support system and the feedback loop. It is a **proposed deployment architecture**; the notebook implements data preparation, model training, and evaluation, but does not itself deploy a live service.

```mermaid
flowchart TD
    A["1. Data input<br/>Building survey and location data"] --> B["2. Data validation and preprocessing<br/>Schema checks, missing values, encoding"]
    B --> C["3. Prediction engine<br/>Validation-selected trained classifier"]
    C --> D["4. Knowledge and decision support<br/>Risk estimate, explanation, uncertainty"]
    D --> E["Human review and action<br/>Qualified engineer / authorised agency"]
    E --> F["5. Feedback, monitoring and governance<br/>Verified outcomes, drift and subgroup checks"]
    F -. "Approved updates only" .-> B
```

### Explanation of the five elements

1. **Data input:** Collect building survey information and relevant geographic/context fields. Validate data provenance, permissions, completeness, and the time at which each field becomes available.
2. **Preprocessing and feature management:** Apply the same approved feature schema and preprocessing steps used during training. Check missing fields, invalid values, unseen categories, and data quality before prediction.
3. **Prediction engine:** Use the model selected by validation macro F1 and then evaluated on the held-out test set. Return a risk estimate and model version, not just an unexplained label.
4. **Knowledge and decision support:** Present the estimate with uncertainty, relevant contributing factors, and clear limitations. Apply risk thresholds only with domain-expert validation; do not let a model output automatically declare a building safe or unsafe.
5. **Feedback, monitoring and governance:** Have qualified people review consequential decisions. Store verified inspection outcomes where lawful, monitor data drift and performance across districts/building groups, document changes, and retrain only after review and re-validation.

**End-to-end flow:** Building data → validation/preprocessing → trained classifier → risk estimate and explanation → expert review/action → verified feedback and monitoring.


## 13. Ethical governance, limitations, and future work

### Ethical considerations
- **Safety:** predictions must support, not replace, qualified engineering inspection.
- **Fairness:** evaluate performance across districts and building groups so that systematic under-identification is noticed.
- **Privacy:** collect only necessary information, protect identifiable records, and control access.
- **Transparency:** explain the model's purpose, data limitations, and uncertainty to users.
- **Accountability:** keep records of model versions and decisions, provide a route for human review, and assign responsibility for deployment.

### Limitations
- The binary target combines grades 4 and 5 and grades 1–3, losing detail about individual damage grades.
- Dataset patterns may not generalise to every district, building type, or future earthquake.
- Some useful predictors may be missing, incomplete, or measured inconsistently.
- Feature importance shows model associations, not causation.
- Performance metrics alone do not establish that deployment would improve safety.

### Future work
Test additional models, evaluate calibration and decision thresholds, conduct subgroup evaluation, validate on independent data, and consult disaster-management and structural-engineering experts before any operational use.

## 14. Conclusion and reporting notes

After running the notebook, write a short conclusion using the actual outputs. It should state:

- which model achieved the stronger validation and test performance;
- the selected hyperparameters and test metrics;
- what the confusion matrices reveal about missed severe-damage cases;
- whether the project objectives were met;
- key limitations and the next steps needed before practical use.

Do not invent results or claim that the model is ready for deployment. Include screenshots or exported outputs of the executed notebook in the evidence appendix if required by the assignment brief.

**Evaluation reminder:** In the final report, state which model was selected from validation macro F1, then quote the actual test metrics and severe-class recall produced by the executed notebook. Do not claim a particular model is superior until you have run the notebook and inspected its output. Explain any false negatives and note that the system is a screening aid, not an engineering certification.
